# 05 · Gaussian reference 밖의 구조를 배우는가?
## Matched-moment GMM / 정확한 score / Fourier residual

`fourier-score`의 **실제 `FourierGaussian`, `NoiseProcess`, `training_loss`**를 불러와 작은 합성 실험을 합니다. 신경망만 전체 입력을 보는 작은 MLP로 바꿉니다. **NCSN++ 성능 재현이나 FID 실험은 아닙니다.**

**핵심 질문**

1. 평균과 공분산이 정확히 같은 Gaussian과 비가우시안 GMM에서, Gaussian 기준항이 설명하지 못하는 score를 신경망이 학습합니까?
2. 주파수별 분산이 모두 같으면 Scalar와 Fourier가 일치합니까?
3. 주파수별 분산 차이를 키웠을 때 Fourier의 **추가 이점이 실제로** 나타납니까? 순위는 미리 가정하지 않습니다.

**설계:** population mean/power를 알고 있는 분포에서 fresh samples로 DSM을 학습합니다. Oracle score는 **평가·수치 검증에만** 사용합니다. 방법 사이에 backbone 초기화, 데이터·잡음·시간 난수열, optimizer, EMA, 학습 budget을 맞춥니다.

| 비교군 | scaled score | 목적 |
|:--|:--|:--|
| `score` | $h_\theta$ | 기본 DSM |
| `scalar_gaussian` | $\sigma s_{G,\mathrm{scalar}}+b_{\mathrm{scalar}}h_\theta$ | 주파수별 covariance 제거 |
| `fourier_gaussian_unscaled` | $\sigma s_G+h_\theta$ | 잔차 정규화 제거 |
| `fourier_gaussian` | $\sigma s_G+F^{-1}[bFh_\theta]$ | 전체 구성 |
| `reference_only` | $\sigma s_G$ | 학습하지 않은 분석적 기준선 |

기본값은 **`smoke`**입니다. 짧은 실행의 수치를 논문 결과로 사용하지 마세요. 셀을 위에서부터 실행한 뒤, 본 실험에서는 아래의 `pilot` 또는 `experiment`를 선택해 **처음부터 다시 실행**합니다.

### 저장 위치와 실행

이 파일을 저장소의 `notebooks/05_gmm_fourier_residual.ipynb`에 두세요. 저장소가 없는 위치에서 자동으로 clone하거나 패키지를 설치하지 않습니다.

저장소 루트의 터미널에서:

```bash
mkdir -p notebooks
# 다운로드한 ipynb를 notebooks/에 저장한 뒤:
uv run --extra figures --with jupyterlab --with ipykernel jupyter lab notebooks/05_gmm_fourier_residual.ipynb
```

추가 패키지는 Jupyter와 `matplotlib`뿐이며 표·CSV에는 pandas를 쓰지 않습니다. 결과는 `saved/gmm_oracle/<preset>/<run_tag>/`에 저장됩니다. 기존 이미지 모델 학습/체크포인트는 수정하지 않습니다.

CUDA/CPU를 지원하는 코드입니다. CUDA가 없으면 CPU를 선택합니다. 이 노트북의 float64 oracle 검증 때문에 **MPS는 선택하지 않습니다**. GPU에서 본 실험을 진행하더라도 oracle bank는 CPU float64로 만들고, 학습은 FP32입니다.

In [ ]:
from __future__ import annotations

import os
# CUDA가 이미 초기화된 커널이면 재시작한 뒤 이 셀부터 실행하세요.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import copy
import csv
import hashlib
import html
import json
import math
import platform
import random
import subprocess
import sys
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import HTML, display

NOTEBOOK_NAME = "05_gmm_fourier_residual.ipynb"
NOTEBOOK_VERSION = "1.0.0"
# 자동 탐색이 실패할 때만 경로를 지정하세요.
REPO_ROOT_OVERRIDE = os.environ.get("FOURIER_SCORE_ROOT")  # 예: "/workspace/fourier-score"


def find_repo_root() -> Path:
    candidates = ([Path(REPO_ROOT_OVERRIDE).expanduser()] if REPO_ROOT_OVERRIDE else [])
    cwd = Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, cwd / "fourier-score"]
    for p in candidates:
        if (p / "fourier_score/method.py").is_file() and (p / "fourier_score/loss.py").is_file():
            return p.resolve()
    raise FileNotFoundError(
        "fourier-score 저장소를 찾지 못했습니다. 이 파일을 저장소의 notebooks/에 두거나 "
        "REPO_ROOT_OVERRIDE를 지정해 주세요. 자동 다운로드는 하지 않습니다."
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fourier_score.method import FourierGaussian, GAUSSIAN_OBJECTIVES
from fourier_score.diffusion import NoiseLevel, NoiseProcess
from fourier_score.loss import training_loss
from fourier_score.spectral import conjugate_symmetrize

# 다른 checkout의 패키지가 이미 import된 커널에서 섞여 실행되는 것을 막습니다.
import fourier_score.method as repository_method
assert Path(repository_method.__file__).resolve().parent.parent == ROOT, (
    "다른 checkout이 이미 import되어 있습니다. 커널을 재시작하세요."
)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def git_blob_sha(path: Path) -> str:
    data = path.read_bytes()
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()


def git_output(*args) -> str | None:
    try:
        return subprocess.check_output(
            ["git", "-C", str(ROOT), *args], stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


SOURCE_FILES = [f"fourier_score/{s}.py" for s in ("method", "loss", "diffusion", "spectral")]
SOURCE_HASHES = {p: sha256_file(ROOT / p) for p in SOURCE_FILES}
EXPECTED_BLOBS = {
    "fourier_score/method.py": "3358fe248d5216a6d05a9101fec201e9c344c2a7",
    "fourier_score/loss.py": "e5a5cc3d713cc11b6d7d88033028637ed942d4f4",
    "fourier_score/diffusion.py": "98f98bcee9d6653a67ccee099fcc15e5d1c6dc5a",
    "fourier_score/spectral.py": "320184a2ddb988bcaa22a06f205238729af5a3d1",
}
changed = [p for p, sha in EXPECTED_BLOBS.items() if git_blob_sha(ROOT / p) != sha]
if changed:
    warnings.warn("참고한 버전과 다른 파일이 있습니다. 아래 검증을 확인하세요: " + ", ".join(changed))

notebook_candidates = [ROOT / "notebooks" / NOTEBOOK_NAME, Path.cwd() / NOTEBOOK_NAME]
NOTEBOOK_PATH = next((p for p in notebook_candidates if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError(f"소스 기록을 위해 파일 이름을 {NOTEBOOK_NAME}로 저장해 주세요.")


def notebook_code_hash() -> str:
    saved = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    sources = ["".join(c["source"]) for c in saved["cells"] if c["cell_type"] == "code"]
    return hashlib.sha256("\n\n".join(sources).encode()).hexdigest()


# 실행 전 노트북을 저장하세요. 출력은 hash에서 제외되므로 결과 셀 저장은 resume에 영향이 없습니다.
NOTEBOOK_CODE_SHA = notebook_code_hash()
print("Repository:", ROOT)
print("Commit:", git_output("rev-parse", "HEAD") or "not a git checkout")
print("Python:", platform.python_version(), "| PyTorch:", str(torch.__version__))
print("Repository source matches reference:", not changed)

## 1. 실행 예산을 먼저 고정합니다

`smoke`는 실행 경로 검증, `pilot`은 탐색, `experiment`는 반복 학습용 **제안 설정**입니다. `experiment`라는 이름이 충분한 수렴이나 논문 완성도를 보장하지 않습니다. Pilot에서 budget/학습률을 정했다면 모든 방법에 같은 선택 규칙을 적용하고, test 결과를 보고 튜닝하지 마세요.

`experiment` 기본 조합은 **2개 분포 × 3개 spectrum × 3개 seed × 4개 방법 = 72회 학습**입니다. 먼저 `pilot` 처리시간으로 전체 비용을 추산하세요. 독립 seed로 실행을 나눌 때는 각 프로세스에 다른 `run_tag`를 사용하세요. **같은 output 경로에 여러 커널을 동시에 쓰지 마세요.**

`run_tag`를 바꾸면 새 결과 경로를 만들고, 같은 설정으로 다시 실행하면 마지막 평가 지점에서 재개하거나 완료된 실행을 읽습니다. 다른 설정·소스·runtime은 같은 실행 경로로 덮어쓰지 않습니다.

In [ ]:
@dataclass(frozen=True)
class Config:
    preset: str
    run_tag: str = "v1"
    image_size: int = 8
    rho: float = 0.85
    spectrum_lambdas: tuple[float, ...] = (0.0, 1.0)
    spectrum_knee: float = 0.15
    spectrum_exponent: float = 1.5
    geometry_seed: int = 31415
    distributions: tuple[str, ...] = ("gaussian", "gmm")
    methods: tuple[str, ...] = (
        "score", "scalar_gaussian", "fourier_gaussian_unscaled", "fourier_gaussian"
    )
    seeds: tuple[int, ...] = (42,)
    steps: int = 1500
    eval_every: int = 250
    batch_size: int = 128
    width: int = 128
    depth: int = 3
    time_features: int = 16
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    ema_decay: float = 0.99
    sigma_min: float = 0.10
    sigma_max: float = 3.0
    n_noise_levels: int = 7
    val_per_noise: int = 256
    test_per_noise: int = 1024
    eval_batch_size: int = 128
    frequency_bins: int = 4
    cpu_threads: int = 4
    device: str = "auto"


PRESET = os.environ.get("GMM_PRESET", "smoke")  # "smoke" / "pilot" / "experiment"
PRESETS = {
    "smoke": Config(
        preset="smoke", image_size=4, steps=80, eval_every=40, batch_size=64,
        width=64, depth=2, n_noise_levels=3, val_per_noise=64, test_per_noise=128,
        eval_batch_size=64, frequency_bins=3,
    ),
    "pilot": Config(preset="pilot"),
    "experiment": Config(
        preset="experiment", steps=5000, eval_every=500, width=192,
        seeds=(42, 43, 44), spectrum_lambdas=(0.0, 0.5, 1.0),
        n_noise_levels=9, val_per_noise=512, test_per_noise=2048,
    ),
}
if PRESET not in PRESETS:
    raise ValueError(f"Unknown PRESET: {PRESET}")
CFG = PRESETS[PRESET]
# 필요하면 여기서 한 번만 수정한 뒤, 노트북을 저장하고 처음부터 실행하세요.
# CFG = replace(CFG, run_tag="pilot_lr2e4", learning_rate=2e-4)
# CFG = replace(CFG, run_tag="seed43", seeds=(43,))

assert 2 <= CFG.image_size <= 16
assert 0.0 <= CFG.rho < 1.0
assert CFG.steps >= 1 and CFG.eval_every >= 1 and CFG.batch_size >= 2
assert CFG.sigma_min > 0 and CFG.sigma_max > CFG.sigma_min
assert CFG.n_noise_levels >= 2 and CFG.time_features % 2 == 0
assert CFG.val_per_noise >= 2 and CFG.test_per_noise >= 2
assert all(0 <= x <= 1 for x in CFG.spectrum_lambdas)
assert len(set(CFG.seeds)) == len(CFG.seeds)
assert set(CFG.methods) <= {"score", *GAUSSIAN_OBJECTIVES}
assert set(CFG.distributions) <= {"gaussian", "gmm"}
assert len(set(CFG.methods)) == len(CFG.methods)

DEVICE = torch.device(
    ("cuda" if torch.cuda.is_available() else "cpu") if CFG.device == "auto" else CFG.device
)
if DEVICE.type not in {"cpu", "cuda"}:
    raise ValueError("이 노트북은 CPU/CUDA만 선택합니다. MPS에서는 CPU로 실행하세요.")
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA를 요청했지만 사용할 수 없습니다.")

torch.set_num_threads(CFG.cpu_threads)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

PROCESS_CONFIG = dict(
    type="ve", sigma_min=CFG.sigma_min, sigma_max=CFG.sigma_max,
    num_scales=1000, t_min=0.0, beta_start=1e-4, beta_end=0.02,
)
PROCESS = NoiseProcess(PROCESS_CONFIG)
OUTPUT_ROOT = ROOT / "saved" / "gmm_oracle" / CFG.preset / CFG.run_tag
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def clean_json(x):
    if isinstance(x, dict):
        return {str(k): clean_json(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [clean_json(v) for v in x]
    if isinstance(x, (np.floating, float)):
        return float(x) if math.isfinite(float(x)) else None
    if isinstance(x, np.integer):
        return int(x)
    return x


def atomic_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(clean_json(obj), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    keys = list(dict.fromkeys(k for row in rows for k in row))
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows([{k: clean_json(row.get(k)) for k in keys} for row in rows])
    os.replace(tmp, path)


def show_table(rows, columns=None, max_rows=60):
    if not rows:
        print("No rows.")
        return
    columns = columns or list(rows[0])
    def fmt(v):
        if v is None:
            return "—"
        if isinstance(v, (float, np.floating)):
            return f"{v:.6g}" if np.isfinite(v) else "—"
        return str(v)
    header = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(fmt(r.get(c)))}</td>" for c in columns) + "</tr>" for r in rows[:max_rows])
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    if len(rows) > max_rows:
        print(f"Showing {max_rows}/{len(rows)} rows; complete rows are saved as CSV.")


ENVIRONMENT = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "numpy": str(np.__version__), "platform": platform.platform(),
    "device": str(DEVICE),
    "device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else platform.processor(),
    "cuda_runtime": torch.version.cuda, "cpu_threads": torch.get_num_threads(),
    "deterministic_algorithms": True, "tf32": False,
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_dirty": bool(git_output("status", "--porcelain")),
    "source_sha256": SOURCE_HASHES, "notebook_code_sha256": NOTEBOOK_CODE_SHA,
    "notebook_version": NOTEBOOK_VERSION,
}
PLAN = {"config": asdict(CFG), "environment": ENVIRONMENT, "protocol": "population-statistics / online DSM / final-step test"}
plan_file = OUTPUT_ROOT / "plan.json"
if plan_file.exists() and json.loads(plan_file.read_text()) != clean_json(PLAN):
    raise RuntimeError("동일 output 경로에 다른 설정/소스/runtime이 있습니다. run_tag를 변경하세요.")
atomic_json(plan_file, PLAN)
RUN_COUNT = len(CFG.distributions) * len(CFG.spectrum_lambdas) * len(CFG.seeds) * len(CFG.methods)
print(f"Preset={CFG.preset}; device={DEVICE}; {RUN_COUNT} runs × {CFG.steps} updates")
print("Output:", OUTPUT_ROOT)
print("SMOKE: 실행 점검용이며 성능 결론을 내리지 않습니다." if PRESET == "smoke" else "설정은 실행 전 고정하고 test로 튜닝하지 마세요.")

## 2. 같은 평균·공분산을 갖는 분포를 구성합니다

한 채널의 $H\times H$ 격자를 $d=H^2$차원 벡터로 봅니다. 정규직교 FFT $F$와 양수 spectrum $P$에 대해

\[
C=F^{-1}\operatorname{diag}(P)F,\qquad L=C^{1/2}
\]

로 정의합니다. $P$는 실수 데이터의 켤레대칭을 만족합니다. 임의의 **고정된 실수 직교행렬** $Q$의 행을 $q_j$라 하고

\[
a_{j,\pm}=\pm\rho\sqrt d\,q_j,\quad
m_{j,\pm}=L a_{j,\pm},\quad
\Sigma_{\rm within}=(1-\rho^2)C
\]

로 놓습니다. $2d$개 성분에 동일 가중치를 주면

\[
p_{\rm GMM}=\frac1{2d}\sum_{j,\pm}\mathcal N(m_{j,\pm},(1-\rho^2)C),
\quad \mathbb E[X]=0,\quad \operatorname{Cov}(X)=C.
\]

비교 Gaussian은 $p_G=\mathcal N(0,C)$입니다. **두 분포의 covariance 전체가 정확히 같습니다.** 따라서 Gaussian reference도 같습니다. 데이터 수 부족 없이도 GMM의 joint score에는 일반적으로 잔차가 남습니다.

Spectrum은 평균 파워를 1로 유지하면서
\[
P_k(\lambda)=(1-\lambda)+\lambda P_k^{\rm structured},\qquad
\operatorname{mean}_kP_k^{\rm structured}=1
\]
로 조절합니다. $\lambda=0$에서는 Scalar와 Fourier가 같은 방법입니다. $\lambda$가 커질 때 이득이 커지는지는 **실험 질문**이지 보장이 아닙니다. `rho`는 비가우시안 구조의 세기를 조절합니다.

회전 $Q$는 모든 학습 seed에 공통입니다. 여기서 seed 반복은 **고정된 합성 분포의 학습 변동**을 측정합니다. 다양한 mixture geometry로 일반화하려면 `geometry_seed`를 별도의 실험 축으로 추가해야 합니다.

In [ ]:
def fft_filter(x: torch.Tensor, multiplier: torch.Tensor) -> torch.Tensor:
    """Orthonormal full-FFT real filter; valid for conjugate-symmetric multipliers."""
    return torch.fft.ifft2(torch.fft.fft2(x, norm="ortho") * multiplier, norm="ortho").real


def make_power(size: int, lam: float, cfg: Config = CFG) -> torch.Tensor:
    f = torch.fft.fftfreq(size, dtype=torch.float64)
    radius = torch.sqrt(f[:, None].square() + f[None, :].square())
    structured = (1.0 + (radius / cfg.spectrum_knee).square()).pow(-cfg.spectrum_exponent)
    structured = conjugate_symmetrize(structured)
    structured = structured / structured.mean()
    power = (1.0 - lam) * torch.ones_like(structured) + lam * structured
    assert float(power.min()) > 1e-4, "population 실험에서 power flooring이 개입하지 않게 하세요."
    return power


class MatchedMomentFamily:
    """Small real Gaussian / equal-weight common-covariance GMM, with exact moments.

    Sampling: dense real square root built from the orthonormal Fourier operator.
    Oracle: independent real eigencoordinates, never marginal Fourier scores.
    """
    def __init__(self, cfg: Config, distribution: str, lam: float):
        self.cfg = cfg
        self.distribution = distribution
        self.lam = float(lam)
        self.size = cfg.image_size
        self.d = self.size ** 2
        self.rho = 0.0 if distribution == "gaussian" else cfg.rho
        self.power = make_power(self.size, self.lam, cfg)
        eye_images = torch.eye(self.d, dtype=torch.float64).reshape(self.d, self.size, self.size)
        # Each row is the filtered basis vector; the filter matrix is real symmetric.
        self.root = fft_filter(eye_images, self.power.sqrt()).reshape(self.d, self.d)
        self.covariance = self.root.T @ self.root
        self.within_fraction = 1.0 - self.rho ** 2
        if distribution == "gaussian":
            self.means = torch.zeros(1, self.d, dtype=torch.float64)
        elif distribution == "gmm":
            generator = torch.Generator().manual_seed(cfg.geometry_seed)
            matrix = torch.randn(self.d, self.d, generator=generator, dtype=torch.float64)
            q, r = torch.linalg.qr(matrix)
            # Fix the conventional QR signs for a reproducible geometry.
            q = q * torch.where(torch.diag(r) < 0, -1.0, 1.0)[None, :]
            centers = self.rho * math.sqrt(self.d) * torch.cat((q, -q), dim=0)
            self.means = centers @ self.root
        else:
            raise ValueError(distribution)
        self.n_components = len(self.means)
        self.eigenvalues, self.eigenvectors = torch.linalg.eigh(self.covariance)
        if self.eigenvalues.min() <= 0:
            raise ValueError("Covariance must be positive definite.")
        self.means_eigen = self.means @ self.eigenvectors
        self.stats = {"mean": torch.zeros(1, self.size, self.size), "power": self.power[None].float()}
        self._means32 = self.means.float()
        self._root32 = self.root.float()
        self.case_id = f"{distribution}_lambda{self.lam:g}".replace(".", "p")

    def sample_cpu(self, n: int, generator: torch.Generator, dtype=torch.float32) -> torch.Tensor:
        # One CPU generator isolates training data from network/evaluation randomness.
        index = torch.randint(self.n_components, (n,), generator=generator)
        z = torch.randn(n, self.d, generator=generator, dtype=dtype)
        means = self._means32 if dtype == torch.float32 else self.means.to(dtype)
        root = self._root32 if dtype == torch.float32 else self.root.to(dtype)
        x = means[index] + math.sqrt(self.within_fraction) * (z @ root)
        return x.reshape(n, 1, self.size, self.size)

    def analytic_moments(self):
        mean = self.means.mean(0)
        centered = self.means - mean
        covariance = centered.T @ centered / self.n_components + self.within_fraction * self.covariance
        return mean, covariance

    def log_prob_and_score(self, y: torch.Tensor, alpha: torch.Tensor, sigma: torch.Tensor):
        """Exact JOINT noisy density/score for each (y, alpha, sigma).

        y: [B,1,H,H]; alpha/sigma: [B]. This function is differentiable,
        but is used only to build validation/test banks and test the oracle.
        All mixture responsibilities condition on the ENTIRE y.
        """
        if y.ndim != 4 or y.shape[1:] != (1, self.size, self.size):
            raise ValueError("Expected [B,1,H,H].")
        if alpha.shape != (len(y),) or sigma.shape != (len(y),):
            raise ValueError("alpha and sigma must have shape [B].")
        if bool((sigma <= 0).any()):
            raise ValueError("sigma must be positive.")
        u = self.eigenvectors.to(y)
        eigenvalues = self.eigenvalues.to(y)
        means_eigen = self.means_eigen.to(y)
        y_eigen = y.flatten(1) @ u
        variance = alpha[:, None].square() * self.within_fraction * eigenvalues[None] + sigma[:, None].square()
        delta = y_eigen[:, None, :] - alpha[:, None, None] * means_eigen[None]
        log_components = -0.5 * (
            (delta.square() / variance[:, None, :]).sum(-1)
            + variance.log().sum(-1)[:, None] + self.d * math.log(2.0 * math.pi)
        ) - math.log(self.n_components)
        log_density = torch.logsumexp(log_components, dim=1)
        responsibility = torch.softmax(log_components, dim=1)
        score_eigen = -(responsibility[:, :, None] * delta / variance[:, None, :]).sum(1)
        score = (score_eigen @ u.T).reshape_as(y)
        return log_density, score

    def gaussian_score(self, y, alpha, sigma, scalar=False):
        power = self.power.to(y)
        if scalar:
            power = power.mean().expand_as(power)
        denom = alpha[:, None, None, None].square() * power + sigma[:, None, None, None].square()
        return -fft_filter(y, denom.reciprocal())


FAMILIES = {
    (kind, float(lam)): MatchedMomentFamily(CFG, kind, float(lam))
    for kind in CFG.distributions for lam in CFG.spectrum_lambdas
}
show_table([
    dict(distribution=f.distribution, spectrum_lambda=f.lam, dimension=f.d,
         components=f.n_components, power_mean=float(f.power.mean()),
         power_min=float(f.power.min()), power_max=float(f.power.max()),
         spectral_condition=float(f.power.max() / f.power.min()))
    for f in FAMILIES.values()
])

## 3. 학습 전에 반드시 통과해야 하는 수치 검증

Oracle의 공식은 다음입니다. $\gamma_j(y,t)$는 noisy GMM의 **전체 관측값**에 대한 성분 responsibility입니다.

\[
C_{j,t}=\alpha_t^2\Sigma_j+\sigma_t^2I,\quad
s_*(y,t)=-\sum_j\gamma_j(y,t)C_{j,t}^{-1}(y-\alpha_tm_j).
\]

실제 구현에서는 공통 within-covariance를 실수 고유좌표로 대각화하고 log-sum-exp로 density를 계산합니다. 별도의 dense `MultivariateNormal` 밀도의 autograd 결과와 대조합니다. 이 검증에는 $\alpha\ne1$도 포함하지만, 아래 **학습 실험 자체는 VE**입니다.

또한 Gaussian의 이상적인 residual이 0이라고 해서 DSM 학습 loss가 0이어야 하는 것은 아닙니다. finite minibatch 학습은 정확한 Gaussian reference에 오히려 작은 학습 오차를 추가할 수도 있습니다.

In [ ]:
def run_numerical_tests() -> list[dict]:
    checks = []
    def record(name, value=0.0):
        checks.append({"test": name, "status": "PASS", "max_error": float(value)})

    generator = torch.Generator().manual_seed(271828)
    size = min(CFG.image_size, 4)
    small = replace(CFG, image_size=size)
    gaussian = MatchedMomentFamily(small, "gaussian", 1.0)
    mixture = MatchedMomentFamily(small, "gmm", 1.0)
    flat = MatchedMomentFamily(small, "gmm", 0.0)

    for family in (gaussian, mixture, flat):
        mean, covariance = family.analytic_moments()
        err = (covariance - family.covariance).abs().max()
        torch.testing.assert_close(mean, torch.zeros_like(mean), atol=2e-12, rtol=0)
        torch.testing.assert_close(covariance, family.covariance, atol=2e-11, rtol=2e-11)
        torch.testing.assert_close(family.power, conjugate_symmetrize(family.power), atol=1e-14, rtol=0)
        record(f"population moments / {family.case_id}", err)
    torch.testing.assert_close(gaussian.stats["power"], mixture.stats["power"], atol=0, rtol=0)
    record("Gaussian and GMM use identical reference statistics")

    y = torch.randn(4, 1, size, size, generator=generator, dtype=torch.float64)
    alpha = torch.tensor([1.0, 0.8, 0.65, 0.4], dtype=torch.float64)
    sigma = torch.tensor([0.15, 0.4, 0.8, 1.5], dtype=torch.float64)
    y_grad = y.clone().requires_grad_(True)
    log_oracle, score_oracle = mixture.log_prob_and_score(y_grad, alpha, sigma)
    covariance = (alpha[:, None, None].square() * mixture.within_fraction * mixture.covariance
                  + sigma[:, None, None].square() * torch.eye(mixture.d, dtype=torch.float64))
    dense = torch.distributions.MultivariateNormal(
        loc=alpha[:, None, None] * mixture.means[None],
        covariance_matrix=covariance[:, None],
    )
    log_dense = torch.logsumexp(dense.log_prob(y_grad.flatten(1)[:, None]), dim=1) - math.log(mixture.n_components)
    grad_dense, = torch.autograd.grad(log_dense.sum(), y_grad)
    torch.testing.assert_close(log_oracle, log_dense, atol=2e-9, rtol=2e-9)
    torch.testing.assert_close(score_oracle, grad_dense, atol=2e-9, rtol=2e-9)
    record("joint oracle = dense log-density autograd (nontrivial alpha)", (score_oracle - grad_dense).abs().max().detach())

    _, gaussian_oracle = gaussian.log_prob_and_score(y, alpha, sigma)
    ref = FourierGaussian(gaussian.stats, backend="fft").double()
    reference = ref.scaled_score(torch.zeros_like(y), y, alpha, sigma) / sigma[:, None, None, None]
    torch.testing.assert_close(reference, gaussian_oracle, atol=8e-6, rtol=2e-6)
    record("Gaussian oracle = repository Fourier reference", (reference - gaussian_oracle).abs().max())

    raw_f = torch.randn(y.shape, generator=generator, dtype=torch.float64, requires_grad=True)
    raw_s = raw_f.detach().clone().requires_grad_(True)
    rf = FourierGaussian(flat.stats, backend="fft").double()
    rs = FourierGaussian(flat.stats, backend="fft", covariance="scalar").double()
    out_f = rf.scaled_score(raw_f, y, alpha, sigma)
    out_s = rs.scaled_score(raw_s, y, alpha, sigma)
    grad_f, = torch.autograd.grad(out_f.square().sum(), raw_f)
    grad_s, = torch.autograd.grad(out_s.square().sum(), raw_s)
    torch.testing.assert_close(out_f, out_s, atol=2e-12, rtol=2e-12)
    torch.testing.assert_close(grad_f, grad_s, atol=2e-12, rtol=2e-12)
    record("flat spectrum: Scalar/Fourier outputs AND gradients", (out_f - out_s).abs().max().detach())

    e = torch.randn(y.shape, generator=generator, dtype=torch.float64)
    pixel = e.square().flatten(1).mean(1)
    spectral = torch.fft.fft2(e, norm="ortho").abs().square().flatten(1).mean(1)
    torch.testing.assert_close(pixel, spectral, atol=2e-12, rtol=2e-12)
    record("full orthonormal FFT Parseval", (pixel - spectral).abs().max())

    # Analytic target variance for an arbitrary reference variance q, not a sample estimate.
    p = mixture.power
    s2 = 0.7 ** 2
    q = torch.full_like(p, float(p.mean()))
    r_q = (s2 * p + q.square()) / (q + s2).square()
    r_p = p / (p + s2)
    gap = s2 * (q - p).square() / ((q + s2).square() * (p + s2))
    torch.testing.assert_close(r_q - r_p, gap, atol=2e-12, rtol=2e-12)
    record("reference covariance mismatch identity", (r_q - r_p - gap).abs().max())
    return checks


NUMERICAL_TESTS = run_numerical_tests()
show_table(NUMERICAL_TESTS)
atomic_json(OUTPUT_ROOT / "numerical_tests.json", NUMERICAL_TESTS)

## 4. 레포의 DSM·adapter + 같은 작은 backbone

학습에는 아래 공통 loss만 사용합니다.

\[
\mathcal L=\mathbb E\,\operatorname{mean}_{\rm pixels}|\sigma s_\theta(y,t)+\epsilon|^2.
\]

즉, oracle를 label로 넣거나 normalized residual에 unweighted MSE를 새로 적용하지 않습니다. Fourier의 $b_k^2$ weighting은 레포의 adapter를 통해 그대로 유지됩니다.

Backbone은 전체 격자를 입력받는 MLP입니다. **주파수별 독립 신경망이 아니며 input whitening도 없습니다.** 마지막 선형층을 0으로 초기화하므로 Gaussian arm은 step 0에서 reference-only로 시작합니다. 같은 초기 backbone이 같은 초기 score를 뜻하지 않는다는 점을 step-0 평가에 그대로 드러냅니다.

Adam, gradient clipping, EMA는 모든 arm에 동일합니다. MLP toy 결과만으로 NCSN++나 실제 이미지의 성능 향상을 주장하지 마세요.

In [ ]:
class SmallJointMLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        frequencies = 2.0 ** torch.arange(cfg.time_features // 2, dtype=torch.float32)
        self.register_buffer("frequencies", frequencies, persistent=False)
        layers = []
        n_in = cfg.image_size ** 2 + cfg.time_features
        for _ in range(cfg.depth):
            layers.extend([nn.Linear(n_in, cfg.width), nn.SiLU()])
            n_in = cfg.width
        layers.append(nn.Linear(n_in, cfg.image_size ** 2))
        self.net = nn.Sequential(*layers)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, y, sigma):
        normalized_time = (sigma.log() - math.log(self.cfg.sigma_min)) / math.log(self.cfg.sigma_max / self.cfg.sigma_min)
        phase = math.pi * normalized_time[:, None] * self.frequencies[None]
        embedding = torch.cat([phase.sin(), phase.cos()], dim=1)
        return self.net(torch.cat([y.flatten(1), embedding], dim=1)).reshape_as(y)


class ToyScoreModel(nn.Module):
    def __init__(self, cfg: Config, stats: dict, method: str):
        super().__init__()
        self.method = method
        self.process = NoiseProcess(PROCESS_CONFIG)
        self.backbone = SmallJointMLP(cfg)
        self.reference = None
        if method in GAUSSIAN_OBJECTIVES:
            self.reference = FourierGaussian(stats, backend="fft", **GAUSSIAN_OBJECTIVES[method])
        elif method != "score":
            raise ValueError(method)

    def scaled_score(self, y, level):
        raw = self.backbone(y, level.sigma)
        if self.reference is None:
            return raw
        return self.reference.scaled_score(raw, y, level.alpha, level.sigma)

    def forward(self, y, level):
        return self.scaled_score(y, level) / level.sigma[:, None, None, None]


def tensor_state_hash(state: dict) -> str:
    h = hashlib.sha256()
    for name, value in sorted(state.items()):
        h.update(name.encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def initial_backbone_state(seed: int):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        backbone = SmallJointMLP(CFG)
    return {k: v.detach().clone() for k, v in backbone.state_dict().items()}


INITIAL_STATES = {seed: initial_backbone_state(seed) for seed in CFG.seeds}
INITIAL_HASHES = {seed: tensor_state_hash(state) for seed, state in INITIAL_STATES.items()}


def make_model(family, method, seed):
    # The global RNG does not control the data stream. No stochastic layers are used.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = ToyScoreModel(CFG, family.stats, method)
    model.backbone.load_state_dict(INITIAL_STATES[seed], strict=True)
    assert tensor_state_hash(model.backbone.state_dict()) == INITIAL_HASHES[seed]
    return model.to(DEVICE)


# Verify that training really calls the repository's objective, including reduction.
_example_family = next(iter(FAMILIES.values()))
_example = make_model(_example_family, "fourier_gaussian", CFG.seeds[0])
_gen = torch.Generator().manual_seed(123)
_x = _example_family.sample_cpu(8, _gen).to(DEVICE)
_lev = PROCESS.sample(8, DEVICE, _gen)
_noise = torch.randn(_x.shape, generator=_gen).to(DEVICE)
_y = PROCESS.perturb(_x, _lev, _noise)
_manual = (_example.scaled_score(_y, _lev) + _noise).square().mean()
_actual = training_loss(_example, _x, _lev, _noise, reduction="mean")
torch.testing.assert_close(_manual, _actual, atol=1e-7, rtol=1e-6)
print("Backbone parameters:", sum(p.numel() for p in _example.backbone.parameters()))
print("Repository DSM equivalence: PASS")
del _example, _x, _lev, _noise, _y, _manual, _actual

## 5. 고정된 검증 bank와 최종 test bank

주 지표는 oracle와의 **scaled true-score error**입니다.

\[
E_\theta=\mathbb E_{t,y}\frac1d\|\sigma_t(s_\theta(y,t)-s_*(y,t))\|^2.
\]

$t$는 log-$\sigma$의 같은 간격 midpoint grid로 적분을 근사합니다. 따라서 아래 수치는 명시된 grid에서의 평균이며, continuous-time 적분을 정확히 계산한 값은 아닙니다. Gaussian reference의 오차 $E_G$와의 비율 $E_\theta/E_G<1$은 그 bank에서 Gaussian 기준항보다 정확해졌다는 뜻입니다. $E_G\simeq0$인 Gaussian에서는 이 비율을 정의하지 않습니다.

참고로 $\|[\sigma s_\theta-\sigma s_G]-[\sigma s_*-\sigma s_G]\|^2$는 **같은 true-score error**입니다. 이를 독립적인 두 성공 지표로 세지 않습니다. residual cosine은 보충 진단입니다.

검증 bank는 학습 곡선, 독립 test bank는 **미리 정한 최종 step**에만 사용합니다. best-test checkpoint 선택은 하지 않습니다. 방법·학습 seed 사이에는 동일한 bank를 재사용하므로, 표의 seed 표준편차는 학습 변동을 나타내며 독립적인 test 표본 변동 전체를 나타내지 않습니다.

In [ ]:
@dataclass
class OracleBank:
    split: str
    entries: list[dict]
    fingerprint: str
    n_observations: int


def bank_seed(family, split: str) -> int:
    payload = f"gmm-oracle-v1:{family.case_id}:{split}:independent-evaluation"
    return 1000000 + int(hashlib.sha256(payload.encode()).hexdigest()[:8], 16)


@torch.no_grad()
def make_bank(family: MatchedMomentFamily, split: str) -> OracleBank:
    n = CFG.val_per_noise if split == "validation" else CFG.test_per_noise
    generator = torch.Generator().manual_seed(bank_seed(family, split))
    entries = []
    digest = hashlib.sha256()
    # Midpoint quadrature in log sigma, not a hand-picked high-performing noise level.
    t_grid = (torch.arange(CFG.n_noise_levels, dtype=torch.float32) + 0.5) / CFG.n_noise_levels
    for noise_bin, t in enumerate(t_grid):
        coordinate = torch.full((n,), float(t))
        level = PROCESS.level(coordinate)
        clean = family.sample_cpu(n, generator)
        noise = torch.randn(clean.shape, generator=generator)
        y = PROCESS.perturb(clean, level, noise)  # input is exactly the FP32 value seen by the model
        oracle_parts, reference_parts, scalar_parts = [], [], []
        for lo in range(0, n, CFG.eval_batch_size):
            hi = min(lo + CFG.eval_batch_size, n)
            yy = y[lo:hi].double()
            aa, ss = level.alpha[lo:hi].double(), level.sigma[lo:hi].double()
            _, exact = family.log_prob_and_score(yy, aa, ss)
            ref = family.gaussian_score(yy, aa, ss)
            scalar = family.gaussian_score(yy, aa, ss, scalar=True)
            oracle_parts.append(ss[:, None, None, None] * exact)
            reference_parts.append(ss[:, None, None, None] * ref)
            scalar_parts.append(ss[:, None, None, None] * scalar)
        entry = {
            "noise_bin": noise_bin, "sigma": float(level.sigma[0]), "coordinate": float(t),
            "y": y, "noise": noise, "oracle_scaled": torch.cat(oracle_parts),
            "reference_scaled": torch.cat(reference_parts), "scalar_reference_scaled": torch.cat(scalar_parts),
        }
        for key in ("y", "noise", "oracle_scaled", "reference_scaled", "scalar_reference_scaled"):
            if not torch.isfinite(entry[key]).all():
                raise FloatingPointError(f"Nonfinite {key} in {family.case_id} / {split}")
            digest.update(entry[key].contiguous().numpy().tobytes())
        digest.update(np.asarray([entry["sigma"], entry["coordinate"]], dtype=np.float64).tobytes())
        entries.append(entry)
    return OracleBank(split, entries, digest.hexdigest(), n * len(entries))


_freq = torch.fft.fftfreq(CFG.image_size, dtype=torch.float64)
_radius = torch.sqrt(_freq[:, None].square() + _freq[None, :].square())
BAND_IDS = (_radius / math.sqrt(0.5) * CFG.frequency_bins).long().clamp_max(CFG.frequency_bins - 1)
BAND_MASKS = [(BAND_IDS == j) for j in range(CFG.frequency_bins)]
BAND_COUNTS = [int(m.sum()) for m in BAND_MASKS]


@torch.no_grad()
def evaluate_model(model: ToyScoreModel | None, family, bank: OracleBank, reference="fourier") -> dict:
    """All reported numerical errors are accumulated on CPU float64.

    model=None evaluates an exact analytic reference (no learned parameters).
    """
    if model is not None:
        model.eval()
    rows = []
    total_error = total_ref = total_dsm = total_dot = total_rhat = 0.0
    total_n = 0
    total_band = np.zeros(CFG.frequency_bins, dtype=np.float64)
    for entry in bank.entries:
        predictions = []
        y_all = entry["y"]
        if model is None:
            key = "reference_scaled" if reference == "fourier" else "scalar_reference_scaled"
            predicted = entry[key]
        else:
            for lo in range(0, len(y_all), CFG.eval_batch_size):
                y = y_all[lo:lo + CFG.eval_batch_size].to(DEVICE)
                # Reuse the EXACT stored sigma. Recomputing exp(log_sigma) on CUDA
                # can differ from the CPU bank by an ulp and alter the oracle comparison.
                level = NoiseLevel(
                    alpha=torch.ones(len(y), device=DEVICE),
                    sigma=torch.full((len(y),), entry["sigma"], device=DEVICE),
                    coordinate=torch.full((len(y),), entry["coordinate"], device=DEVICE),
                )
                predictions.append(model.scaled_score(y, level).detach().cpu().double())
            predicted = torch.cat(predictions)
        true = entry["oracle_scaled"]
        error = predicted - true
        rstar = true - entry["reference_scaled"]
        rhat = predicted - entry["reference_scaled"]
        mse = error.square().flatten(1).mean(1)
        reference_error = rstar.square().flatten(1).mean(1)
        noisy_dsm = (predicted + entry["noise"].double()).square().flatten(1).mean(1)
        dot = (rhat * rstar).flatten(1).mean(1)
        rhat_energy = rhat.square().flatten(1).mean(1)
        energies = torch.fft.fft2(error, norm="ortho").abs().square().mean(dim=(0, 1))
        band_values = [float(energies[mask].mean()) if count else None for mask, count in zip(BAND_MASKS, BAND_COUNTS)]
        n = len(mse)
        err_value, ref_value = float(mse.mean()), float(reference_error.mean())
        row = {
            "noise_bin": entry["noise_bin"], "sigma": entry["sigma"], "n": n,
            "score_error": err_value,
            "unweighted_score_error": err_value / entry["sigma"] ** 2,
            "reference_error": ref_value,
            "relative_to_gaussian": err_value / ref_value if ref_value > 1e-10 else None,
            "dsm_pixel_mean": float(noisy_dsm.mean()),
            **{f"frequency_band_{i}": v for i, v in enumerate(band_values)},
        }
        # Check weighting of full-FFT band diagnostics against canonical pixel mean.
        reconstructed = sum((v or 0.0) * c for v, c in zip(band_values, BAND_COUNTS)) / family.d
        if not math.isclose(reconstructed, err_value, rel_tol=2e-8, abs_tol=2e-11):
            raise AssertionError("Frequency-band means do not reproduce pixel score error.")
        rows.append(row)
        total_n += n
        total_error += float(mse.sum())
        total_ref += float(reference_error.sum())
        total_dsm += float(noisy_dsm.sum())
        total_dot += float(dot.sum())
        total_rhat += float(rhat_energy.sum())
        total_band += n * np.asarray([0.0 if v is None else v for v in band_values])
    if not np.isfinite([total_error, total_ref, total_dsm, total_dot, total_rhat]).all():
        raise FloatingPointError("Nonfinite evaluation metric.")
    denominator = math.sqrt(max(total_ref * total_rhat, 0.0))
    return {
        "split": bank.split, "n_observations": total_n, "bank_sha256": bank.fingerprint,
        "score_error": total_error / total_n,
        "reference_error": total_ref / total_n,
        "relative_to_gaussian": total_error / total_ref if total_ref / total_n > 1e-10 else None,
        "residual_cosine": total_dot / denominator if denominator > 1e-10 else None,
        "dsm_pixel_mean": total_dsm / total_n,
        "score_error_by_frequency": [float(total_band[i] / total_n) if BAND_COUNTS[i] else None for i in range(CFG.frequency_bins)],
        "per_noise": rows,
    }


atomic_json(OUTPUT_ROOT / "frequency_bands.json", {
    "modes_per_channel": BAND_COUNTS,
    "edges_cycles_per_pixel": np.linspace(0, math.sqrt(0.5), CFG.frequency_bins + 1).tolist(),
    "definition": "Full orthonormal FFT; DC and both conjugate partners; mode-count weighting.",
})
print("Band mode counts:", BAND_COUNTS, "sum=", sum(BAND_COUNTS))

## 6. 짝지어진 학습·EMA·중단 재개

동일 `(분포, lambda, seed)`의 각 방법은 **같은 초기 backbone state와 같은 data/t/noise stream**을 사용합니다. 학습에는 매 update 새로운 표본을 생성합니다. finite dataset memorization 실험이 아닙니다.

각 실행은 `checkpoint.pt`, `metrics.json`을 저장합니다. 체크포인트에는 optimizer, raw model, EMA, CPU 데이터 RNG, 마지막 완료 step, 곡선, source/config fingerprint가 포함됩니다. **평가 간격 사이에서 중단되면 마지막으로 저장된 완료 지점부터 재개**합니다. 다른 설정을 덮어쓰지 않습니다. 체크포인트는 본 노트북이 로컬에서 만든 것만 읽으세요.

시간은 data sampling + training + EMA를 평가 구간별로 동기화해서 합산한 `optimizer_seconds`와 `evaluation_seconds`로 나눕니다. Oracle bank 준비, 저장, 시작 비용은 별도입니다. 이 toy MLP의 시간으로 실제 NCSN++의 비용을 추정하지 않습니다.

아래 실행 셀은 **실제로 학습을 시작합니다**. `smoke`를 먼저 확인하세요.

In [ ]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


@torch.no_grad()
def update_ema(ema, model, decay):
    for pe, p in zip(ema.backbone.parameters(), model.backbone.parameters()):
        pe.lerp_(p, 1.0 - decay)


def to_cpu_tree(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_cpu_tree(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_cpu_tree(v) for v in obj)
    return obj


def run_signature(family, method, seed, validation, test):
    description = {
        "config": asdict(CFG), "family": family.case_id, "method": method, "seed": seed,
        "initial_backbone_sha256": INITIAL_HASHES[seed],
        "source_sha256": SOURCE_HASHES, "notebook_code_sha256": NOTEBOOK_CODE_SHA,
        "environment": ENVIRONMENT,
        "validation_sha256": validation.fingerprint, "test_sha256": test.fingerprint,
    }
    return hashlib.sha256(json.dumps(description, sort_keys=True).encode()).hexdigest()


def train_one(family, method, seed, validation, test, *, stop_after=None):
    """stop_after is a QA hook; normal experiment calls leave it None.

    The stop is at a fully saved update boundary and is not a scientific
    checkpoint selection rule. Main runs always test at CFG.steps.
    """
    run_dir = OUTPUT_ROOT / family.case_id / f"{method}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "checkpoint.pt"
    signature = run_signature(family, method, seed, validation, test)
    model = make_model(family, method, seed)
    ema = copy.deepcopy(model).eval()
    ema.requires_grad_(False)
    optimizer = torch.optim.Adam(
        model.backbone.parameters(), lr=CFG.learning_rate,
        betas=(0.9, 0.999), eps=1e-8, weight_decay=CFG.weight_decay,
    )
    generator = torch.Generator().manual_seed(100000 + seed)
    state = {
        "signature": signature, "case_id": family.case_id,
        "distribution": family.distribution, "spectrum_lambda": family.lam,
        "method": method, "seed": int(seed), "step": 0, "completed": False,
        "initial_backbone_sha256": INITIAL_HASHES[seed],
        "optimizer_seconds": 0.0, "evaluation_seconds": 0.0,
        "training_stream_first_batch_sha256": None,
        "validation": [], "test": None,
    }

    def save_state():
        payload = {
            "state": state, "model": model.state_dict(), "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(), "data_rng": generator.get_state(),
            "torch_cpu_rng": torch.get_rng_state(),
        }
        tmp = checkpoint.with_name("checkpoint.pt.tmp")
        torch.save(to_cpu_tree(payload), tmp)
        os.replace(tmp, checkpoint)
        atomic_json(run_dir / "metrics.json", state)

    if checkpoint.exists():
        payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
        if payload["state"]["signature"] != signature:
            raise RuntimeError(f"Configuration/source/bank mismatch at {run_dir}; use a new run_tag.")
        state = payload["state"]
        if state["completed"]:
            print(f"SKIP completed {family.case_id} / {method} / seed {seed}")
            return state
        model.load_state_dict(payload["model"], strict=True)
        ema.load_state_dict(payload["ema"], strict=True)
        optimizer.load_state_dict(payload["optimizer"])
        generator.set_state(payload["data_rng"].cpu())
        torch.set_rng_state(payload["torch_cpu_rng"].cpu())
        print(f"RESUME {family.case_id} / {method} / seed {seed} at {state['step']}")
    else:
        start_eval = time.perf_counter()
        metric = evaluate_model(ema, family, validation)
        state["evaluation_seconds"] += time.perf_counter() - start_eval
        metric.update(step=0, optimizer_seconds=0.0, last_training_loss=None)
        state["validation"].append(metric)
        save_state()

    model.train()
    sync_device()
    block_start = time.perf_counter()
    starting_step = state["step"]
    for step in range(starting_step + 1, CFG.steps + 1):
        clean_cpu = family.sample_cpu(CFG.batch_size, generator)
        level = PROCESS.sample(CFG.batch_size, DEVICE, generator)
        noise_cpu = torch.randn(clean_cpu.shape, generator=generator)
        if step == 1:
            stream_hash = hashlib.sha256()
            for tensor in (clean_cpu, level.coordinate.cpu(), noise_cpu):
                stream_hash.update(tensor.contiguous().numpy().tobytes())
            state["training_stream_first_batch_sha256"] = stream_hash.hexdigest()
        clean, noise = clean_cpu.to(DEVICE), noise_cpu.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(model, clean, level, noise, reduction="mean")
        loss.backward()
        # This checks finite gradients before optimizer.step. All arms share it.
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.backbone.parameters(), CFG.grad_clip, error_if_nonfinite=True
        )
        optimizer.step()
        update_ema(ema, model, CFG.ema_decay)

        should_record = step % CFG.eval_every == 0 or step == CFG.steps or step == stop_after
        if should_record:
            sync_device()
            state["optimizer_seconds"] += time.perf_counter() - block_start
            state["step"] = step
            start_eval = time.perf_counter()
            metric = evaluate_model(ema, family, validation)
            state["evaluation_seconds"] += time.perf_counter() - start_eval
            metric.update(
                step=step, optimizer_seconds=state["optimizer_seconds"],
                last_training_loss=float(loss.detach().cpu()),
                last_gradient_norm=float(gradient_norm.detach().cpu()),
            )
            state["validation"].append(metric)
            save_state()
            print(
                f"{family.case_id:22s} {method:28s} s{seed} {step:5d}/{CFG.steps} "
                f"true-score={metric['score_error']:.5g} "
                f"train_s={state['optimizer_seconds']:.1f}", flush=True
            )
            if stop_after is not None and step == stop_after and step < CFG.steps:
                return state
            model.train()
            sync_device()
            block_start = time.perf_counter()

    # Final predetermined checkpoint, not the best validation/test checkpoint.
    start_eval = time.perf_counter()
    state["test"] = evaluate_model(ema, family, test)
    state["evaluation_seconds"] += time.perf_counter() - start_eval
    state["test"].update(step=CFG.steps, weights="EMA")
    state["final_ema_backbone_sha256"] = tensor_state_hash(ema.backbone.state_dict())
    state["completed"] = True
    save_state()
    return state

In [ ]:
# RUN: execute the planned comparisons. No outcome-dependent stopping or arm removal.
ALL_RESULTS = []
REFERENCES = []
BANK_MANIFESTS = []
experiment_started = time.perf_counter()

for family in FAMILIES.values():
    print(f"\n=== {family.case_id}: preparing independent oracle banks ===", flush=True)
    bank_started = time.perf_counter()
    validation_bank = make_bank(family, "validation")
    test_bank = make_bank(family, "test")
    bank_seconds = time.perf_counter() - bank_started
    BANK_MANIFESTS.append({
        "case_id": family.case_id, "preparation_seconds": bank_seconds,
        "validation_sha256": validation_bank.fingerprint, "test_sha256": test_bank.fingerprint,
        "validation_n": validation_bank.n_observations, "test_n": test_bank.n_observations,
        "validation_seed": bank_seed(family, "validation"), "test_seed": bank_seed(family, "test"),
    })
    for split_bank in (validation_bank, test_bank):
        for reference in ("fourier", "scalar"):
            metric = evaluate_model(None, family, split_bank, reference=reference)
            REFERENCES.append({
                "case_id": family.case_id, "distribution": family.distribution,
                "spectrum_lambda": family.lam, "reference": reference, **metric,
            })

    for seed in CFG.seeds:
        # Rotate execution order deterministically; every arm still receives identical streams.
        methods = list(CFG.methods)
        ordering = np.random.default_rng(seed + 20260922)
        ordering.shuffle(methods)
        paired_states = []
        for method in methods:
            result = train_one(family, method, seed, validation_bank, test_bank)
            if not result["completed"]:
                raise RuntimeError("A partial run cannot enter final results.")
            paired_states.append(result)
            ALL_RESULTS.append(result)
        assert len({r["initial_backbone_sha256"] for r in paired_states}) == 1
        assert len({r["training_stream_first_batch_sha256"] for r in paired_states}) == 1
        # Independent CPU RNGs use identical calls each step; the first batch is also fingerprinted.
    atomic_json(OUTPUT_ROOT / "reference_metrics.json", REFERENCES)
    atomic_json(OUTPUT_ROOT / "oracle_bank_manifests.json", BANK_MANIFESTS)
    atomic_json(OUTPUT_ROOT / "all_results.json", ALL_RESULTS)

SESSION_SECONDS = time.perf_counter() - experiment_started
atomic_json(OUTPUT_ROOT / "session.json", {
    "session_seconds": SESSION_SECONDS, "completed_runs": len(ALL_RESULTS),
    "expected_runs": RUN_COUNT, "note": "Includes bank creation/evaluation/checkpoint I/O; resumed runs may be skipped.",
})
print(f"\nCompleted {len(ALL_RESULTS)}/{RUN_COUNT} runs. This session: {SESSION_SECONDS:.1f} s")
print("Artifacts:", OUTPUT_ROOT)

## 7. 최종 결과표: 어떤 결과가 나와도 그대로 집계합니다

아래 표의 `score_error`는 **final-step held-out test**입니다. `relative_to_gaussian < 1`이면 해당 bank에서 Gaussian 기준항보다 정확합니다. Gaussian reference 오차가 수치적으로 0이면 이 비율은 비워 둡니다.

`seed_sd`는 독립 학습 seed 간 **sample standard deviation**입니다. seed 하나에서는 0으로 표시하지 않고 비워 둡니다. 작은 seed 수의 불확실성이 해소되었다거나 통계적으로 유의하다고 자동 판정하지 않습니다. `paired_delta_F_minus_S < 0`이면 해당 seed에서 Fourier의 error가 낮습니다.

결과가 예상과 달라도 arm/seed/noise bin을 제거하지 마세요. flat spectrum은 차이가 나지 않아야 하는 경계조건이며, Gaussian 학습 결과가 reference-only보다 나쁠 수 있는 것도 숨기지 않습니다.

In [ ]:
FINAL_ROWS = []
NOISE_ROWS = []
CURVE_ROWS = []
for result in ALL_RESULTS:
    common = {k: result[k] for k in ("case_id", "distribution", "spectrum_lambda", "method", "seed")}
    metric = result["test"]
    FINAL_ROWS.append({
        **common, "step": result["step"], "weights": "EMA",
        "score_error": metric["score_error"], "reference_error": metric["reference_error"],
        "relative_to_gaussian": metric["relative_to_gaussian"],
        "residual_cosine": metric["residual_cosine"],
        "dsm_pixel_mean": metric["dsm_pixel_mean"],
        "optimizer_seconds": result["optimizer_seconds"],
        "evaluation_seconds": result["evaluation_seconds"],
        "n_test_observations": metric["n_observations"],
    })
    for row in metric["per_noise"]:
        NOISE_ROWS.append({**common, "split": "test", **row})
    for row in result["validation"]:
        CURVE_ROWS.append({
            **common, "split": "validation", "step": row["step"],
            "score_error": row["score_error"], "relative_to_gaussian": row["relative_to_gaussian"],
            "optimizer_seconds": row["optimizer_seconds"],
            "last_training_loss": row["last_training_loss"],
        })


def summarize(rows, keys, value="score_error"):
    groups = defaultdict(list)
    for row in rows:
        if row.get(value) is not None:
            groups[tuple(row[k] for k in keys)].append(float(row[value]))
    summary = []
    for key, values in sorted(groups.items()):
        values = np.asarray(values)
        summary.append({
            **dict(zip(keys, key)), "metric": value, "n_seeds": len(values),
            "mean": float(values.mean()),
            "seed_sd": float(values.std(ddof=1)) if len(values) > 1 else None,
        })
    return summary


FINAL_SUMMARY = summarize(FINAL_ROWS, ["distribution", "spectrum_lambda", "method"])
paired = defaultdict(dict)
for row in FINAL_ROWS:
    paired[(row["distribution"], row["spectrum_lambda"], row["seed"])][row["method"]] = row["score_error"]
PAIRED_ROWS = []
for (kind, lam, seed), values in sorted(paired.items()):
    if {"fourier_gaussian", "scalar_gaussian"} <= values.keys():
        PAIRED_ROWS.append({
            "distribution": kind, "spectrum_lambda": lam, "seed": seed,
            "paired_delta_F_minus_S": values["fourier_gaussian"] - values["scalar_gaussian"],
        })
PAIRED_SUMMARY = summarize(PAIRED_ROWS, ["distribution", "spectrum_lambda"], value="paired_delta_F_minus_S")
REFERENCE_ROWS = [
    {k: r[k] for k in ("distribution", "spectrum_lambda", "reference", "split", "score_error", "n_observations")}
    for r in REFERENCES
]

write_csv(OUTPUT_ROOT / "final_per_seed.csv", FINAL_ROWS)
write_csv(OUTPUT_ROOT / "final_seed_summary.csv", FINAL_SUMMARY)
write_csv(OUTPUT_ROOT / "paired_F_minus_S.csv", PAIRED_ROWS)
write_csv(OUTPUT_ROOT / "paired_seed_summary.csv", PAIRED_SUMMARY)
write_csv(OUTPUT_ROOT / "validation_curves.csv", CURVE_ROWS)
write_csv(OUTPUT_ROOT / "test_noise_frequency.csv", NOISE_ROWS)
write_csv(OUTPUT_ROOT / "reference_only.csv", REFERENCE_ROWS)

print("Final held-out test — lower true-score error is better")
show_table(FINAL_SUMMARY)
print("Paired Fourier minus Scalar — negative favors Fourier on this metric")
show_table(PAIRED_SUMMARY)
print("Per-seed non-Gaussian residual diagnostics")
show_table([r for r in FINAL_ROWS if r["distribution"] == "gmm"], columns=[
    "spectrum_lambda", "method", "seed", "score_error", "reference_error",
    "relative_to_gaussian", "residual_cosine", "optimizer_seconds",
])

## 8. 그림: 학습 곡선, spectrum sweep, 잡음·주파수별 오차

각 그래프는 별도 figure로 저장합니다. seed 하나에서는 불확실성 band를 만들지 않습니다. reference-only는 학습하지 않는 정확한 기준선입니다. log 축에서 0은 $10^{-12}$로 **표시만** 제한하고, 원본 CSV 수치는 변경하지 않습니다.

시간 그래프는 각 seed의 측정값을 개별 선으로 그립니다. 서로 다른 시간 좌표를 임의로 평균하거나 끝난 실행을 유리하게 연장하지 않습니다. 모델 순위에 맞춰 plot 범위를 선택하지 않고, 사전에 정한 최대 `lambda` GMM 조건을 상세 그림에 사용합니다.

In [ ]:
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
LABELS = {
    "score": "Score DSM", "scalar_gaussian": "Scalar Gaussian",
    "fourier_gaussian_unscaled": "Fourier (unscaled)", "fourier_gaussian": "Fourier Gaussian",
}
PLOT_FLOOR = 1e-12
FOCUS_LAMBDA = max(CFG.spectrum_lambdas)


def finish_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / (filename + ".png"), dpi=160, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / (filename + ".svg"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def grouped_points(rows, xkey, value="score_error"):
    grouped = defaultdict(list)
    for row in rows:
        if row.get(value) is not None:
            grouped[row[xkey]].append(float(row[value]))
    xs = sorted(grouped)
    means = np.asarray([np.mean(grouped[x]) for x in xs])
    sds = np.asarray([np.std(grouped[x], ddof=1) if len(grouped[x]) > 1 else 0.0 for x in xs])
    has_repeats = all(len(grouped[x]) > 1 for x in xs)
    return np.asarray(xs), means, sds, has_repeats


if "gmm" in CFG.distributions:
    focus_curves = [r for r in CURVE_ROWS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in focus_curves if r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "step")
        ax.plot(xs, np.maximum(mean, PLOT_FLOOR), marker="o", label=LABELS[method])
        if repeats:
            ax.fill_between(xs, np.maximum(mean - sd, PLOT_FLOOR), np.maximum(mean + sd, PLOT_FLOOR), alpha=0.15)
    ref = next(r for r in REFERENCES if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["split"] == "validation" and r["reference"] == "fourier")
    ax.axhline(max(ref["score_error"], PLOT_FLOOR), linestyle="--", label="Gaussian reference only")
    ax.set(xlabel="Optimizer updates", ylabel="Scaled true-score MSE", yscale="log",
           title=f"Validation learning curves | GMM | lambda={FOCUS_LAMBDA:g} | {CFG.preset}")
    ax.legend(fontsize=8)
    finish_figure(fig, "01_learning_curves")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        for seed in CFG.seeds:
            rows = sorted((r for r in focus_curves if r["method"] == method and r["seed"] == seed), key=lambda r: r["step"])
            ax.plot([r["optimizer_seconds"] for r in rows], [max(r["score_error"], PLOT_FLOOR) for r in rows],
                    marker=".", label=f"{LABELS[method]} / s{seed}")
    ax.set(xlabel="Cumulative data + optimizer + EMA seconds", ylabel="Scaled true-score MSE", yscale="log",
           title="Measured toy training time (oracle evaluation excluded)")
    ax.legend(fontsize=7)
    finish_figure(fig, "02_learning_vs_time")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in FINAL_ROWS if r["distribution"] == "gmm" and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "spectrum_lambda")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    refs = sorted((r for r in REFERENCES if r["distribution"] == "gmm" and r["split"] == "test" and r["reference"] == "fourier"), key=lambda r: r["spectrum_lambda"])
    ax.plot([r["spectrum_lambda"] for r in refs], [r["score_error"] for r in refs], linestyle="--", label="Gaussian reference only")
    ax.set(xlabel="Spectrum interpolation lambda", ylabel="Final held-out scaled true-score MSE",
           title="Spectrum sweep | mean ± training-seed SD when repeated")
    ax.legend(fontsize=8)
    finish_figure(fig, "03_spectrum_sweep")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in NOISE_ROWS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "sigma")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    ax.set(xlabel="Noise sigma", xscale="log", ylabel="Final held-out scaled true-score MSE",
           title="Noise-resolved true-score error")
    ax.legend(fontsize=8)
    finish_figure(fig, "04_noise_resolved_error")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        selected = [r for r in ALL_RESULTS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["method"] == method]
        values = np.asarray([[np.nan if v is None else v for v in r["test"]["score_error_by_frequency"]] for r in selected])
        mean = values.mean(0)
        sd = values.std(0, ddof=1) if len(values) > 1 else None
        ax.errorbar(np.arange(CFG.frequency_bins), mean, yerr=sd, marker="o", capsize=3, label=LABELS[method])
    ax.set(xlabel="Radial frequency band (DC included)", ylabel="Mean squared Fourier error per mode",
           title="Final true-score error by frequency | not a DSM-noise proxy")
    ax.set_xticks(np.arange(CFG.frequency_bins))
    ax.legend(fontsize=8)
    finish_figure(fig, "05_frequency_resolved_error")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in FINAL_ROWS if r["distribution"] == "gmm" and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "spectrum_lambda", value="relative_to_gaussian")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    ax.axhline(1.0, linestyle="--", label="No improvement over reference")
    ax.set(xlabel="Spectrum interpolation lambda", ylabel="Model error / Gaussian-reference error",
           title="Non-Gaussian residual learning | below 1 means improvement")
    ax.legend(fontsize=8)
    finish_figure(fig, "06_beyond_gaussian_reference")
else:
    print("GMM 조건이 없어 GMM 상세 그림은 생략합니다.")

print("Figures:", FIGURE_DIR)

## 9. 잔차 타깃 모멘트: 별도 Monte Carlo 진단 (학습에 사용하지 않음)

앞의 deterministic 검증에 더해, GMM에서도
\[
T=-\epsilon-\sigma s_G(y),\quad
\mathbb E|F(T)_k|^2=b_k^2=\frac{P_k}{P_k+\sigma^2}
\]
를 표본으로 점검합니다. 이 값은 **개별 noisy regression target**의 모멘트이며, 학습된 residual 출력 자체가 단위 분산이라는 주장이 아닙니다.

아래 표본 추정 오차는 MC 오차를 포함하므로 값이 정확히 1이어야 한다는 assert는 하지 않습니다. 이 진단 역시 모델을 선택하거나 test 결과를 필터링하는 데 사용하지 않습니다.

In [ ]:
@torch.no_grad()
def target_moment_diagnostic(family, n=20000, sigma=0.7, batch_size=512):
    generator = torch.Generator().manual_seed(909090)
    sum_target_power = torch.zeros_like(family.power)
    seen = 0
    while seen < n:
        b = min(batch_size, n - seen)
        x = family.sample_cpu(b, generator, dtype=torch.float64)
        epsilon = torch.randn(x.shape, generator=generator, dtype=torch.float64)
        y = x + sigma * epsilon
        a = torch.ones(b, dtype=torch.float64)
        s = torch.full((b,), sigma, dtype=torch.float64)
        target = -epsilon - sigma * family.gaussian_score(y, a, s)
        sum_target_power += torch.fft.fft2(target, norm="ortho").abs().square().sum(dim=(0, 1))
        seen += b
    empirical = sum_target_power / n
    analytic = family.power / (family.power + sigma ** 2)
    normalized = empirical / analytic
    return {
        "case_id": family.case_id, "n": n, "sigma": sigma,
        "normalized_target_second_moment_mean": float(normalized.mean()),
        "normalized_target_second_moment_min": float(normalized.min()),
        "normalized_target_second_moment_max": float(normalized.max()),
        "power_relative_L2_error": float((empirical - analytic).norm() / analytic.norm()),
    }


MOMENT_ROWS = [target_moment_diagnostic(f, n=4000 if CFG.preset == "smoke" else 20000) for f in FAMILIES.values()]
show_table(MOMENT_ROWS)
write_csv(OUTPUT_ROOT / "target_moment_diagnostic.csv", MOMENT_ROWS)

## 10. 무엇을 논문에 쓸 수 있습니까?

**실제로 확인해야 할 결과:** GMM에서 learned total score가 Gaussian reference-only보다 낮은 **held-out true-score error**를 보이고 반복 seed에서 유지되는지 확인합니다. 두 분포는 covariance 전체가 같고 population 통계를 사용하므로, 그러한 개선은 통계 추정 오차만 보정한 것으로 설명할 수 없습니다. 다만 그것만으로 모든 mode를 완벽하게 학습했다거나 생성 샘플의 품질이 좋아졌다고 결론 내리지는 않습니다.

**Fourier의 추가 효과:** Scalar와 Fourier의 paired 차이를 모든 `lambda`에 대해 보고합니다. `lambda=0`은 수치적으로 같은 방법이어야 합니다. structured spectrum에서도 차이가 없으면 그 결과를 그대로 남겨야 합니다. GMM에서 non-Gaussian residual을 학습하는 것과, 그 일을 Fourier가 Scalar보다 잘하는 것은 **별도의 주장**입니다.

**Gaussian 대조군:** 정확한 reference-only의 true-score error는 수치적으로 0입니다. 유한한 SGD가 여기에 오차를 추가하는 경우를 포함해 보고합니다. 정확한 population 통계에서도 GMM residual은 남으므로, 이 잔차를 “유한 데이터셋 때문에만 생긴 오차”라고 설명하지 않습니다.

**보고 범위:** 작은 joint MLP, VE noise law, 온라인 합성 데이터, population 통계, 고정된 mixture geometry에서의 기제 검증입니다. 이 실험은 CLT 검증, 자연 이미지의 joint Gaussian성 증명, FID/IS 개선 증명, 보편적 최적화 수렴 정리가 아닙니다. 실제 CIFAR/Churches 결과를 대체하지 않습니다.

**요약 파일**

| 파일 | 용도 |
|:--|:--|
| `plan.json` | 설정, 라이브러리, source hash, protocol |
| `numerical_tests.json` | exact-moment / oracle / scalar-equivalence 검증 |
| `final_per_seed.csv`, `final_seed_summary.csv` | final-step held-out true-score error |
| `paired_F_minus_S.csv`, `paired_seed_summary.csv` | 핵심 paired 비교 |
| `validation_curves.csv` | step·측정 시간별 학습 곡선 |
| `test_noise_frequency.csv` | noise×frequency 진단 |
| `reference_only.csv` | 학습하지 않은 Gaussian 기준선 |
| `target_moment_diagnostic.csv` | noisy target 정규화의 MC 점검 |
| `figures/*.png`, `figures/*.svg` | 보고용 그림 |
| `<case>/<method>_seed*/checkpoint.pt` | 동일 설정의 안전한 중단 재개 |

본 실험의 설정을 바꿀 때는 `run_tag`도 바꾸고, 새 test 결과를 비교한 횟수와 튜닝 규칙을 기록하세요. 한 seed에서 `seed_sd`를 0으로 쓰지 말고, 같은 test bank를 쓰는 반복의 한계를 명시하세요.

## 참고 구현과 수학의 범위

본 노트북의 Gaussian/GMM 분포 설계와 oracle 유도는 위 수식에서 직접 검증합니다. GMM 사용 자체나 Gaussian preconditioning 대수의 새로움을 주장하지 않습니다.

- 저장소의 [method.py](https://github.com/junyeopYim/fourier-score/blob/77f856ad7f680cb7f81756a308870e35c61207d1/fourier_score/method.py), [loss.py](https://github.com/junyeopYim/fourier-score/blob/77f856ad7f680cb7f81756a308870e35c61207d1/fourier_score/loss.py), [MATH.md](https://github.com/junyeopYim/fourier-score/blob/77f856ad7f680cb7f81756a308870e35c61207d1/docs/MATH.md): Gaussian reference, output scaling, unchanged DSM.
- [Score-SDE](https://arxiv.org/abs/2011.13456): score-based generative modeling의 배경.
- [EDM](https://arxiv.org/abs/2206.00364): 관련 Gaussian/denoiser preconditioning.
- PyTorch 공식 [FFT normalization](https://docs.pytorch.org/docs/stable/generated/torch.fft.fft2.html), [logsumexp](https://docs.pytorch.org/docs/stable/generated/torch.logsumexp.html), [reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html): 구현상의 FFT 규약, 안정적 mixture density, 재현성의 범위.

실행 시 실제 checkout의 파일 SHA-256과 notebook code hash를 결과와 함께 기록합니다. 장치·라이브러리가 달라져도 bitwise 같다는 보장은 하지 않습니다.

### 제작 시 실행 점검 기록

CPU **Python 3.13.5 / PyTorch 2.10.0+cpu**에서 위 기본 `smoke`의 **16개 실행 × 80 updates**, 표·CSV·그림 저장 경로를 실행 확인했습니다. 저장소에서 읽은 네 핵심 모듈은 Git blob SHA가 일치하는 소스로 검증했습니다.

중간 체크포인트에서 재개한 결과가 중단 없이 학습한 결과와 **model 및 EMA tensor 기준으로 동일**한지 확인했고, 완료된 실행 건너뛰기와 변경된 소스의 재개 거부도 확인했습니다. 이 기록은 실행 검증이며 논문용 성능 증거가 아닙니다. **CUDA 실행과 72회 × 5,000-step 본 실험은 여기서 검증 완료한 것으로 표시하지 않습니다.**

이 배포본의 실행 출력은 비워 두었습니다. 사용하시는 커널에서 저장소 경로·환경 출력을 확인한 뒤 `smoke`부터 다시 실행하세요.